<a href="https://colab.research.google.com/github/NazmulHudaNabil/email-spam-detection/blob/main/Spam_or_not_using_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install nltk

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence

In [ ]:
df = pd.read_csv("/content/spam.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df["Category"])

In [ ]:
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
import string
from nltk.corpus import stopwords

def data_clean(message):
  message_token = word_tokenize(message)
  message_no_punc = [char for char in message_token if char not in string.punctuation]
  message_no_stopwords = [word for word in message_no_punc if word.lower() not in stopwords.words("english")]
  return message_no_stopwords

In [ ]:
tokens = df["Message"].apply(data_clean)

In [ ]:
tokens

In [ ]:
# build vocab

vocab = list(set(word for message in tokens for word in message))

In [ ]:
len(vocab)

In [ ]:
vocab = ["<PAD>", "<UNK>"] + vocab

In [ ]:
word2idx = {word: idx for idx, word in enumerate(vocab)}

In [ ]:
def text_to_indices(message):
    return [word2idx.get(word, word2idx["<UNK>"]) for word in message]

In [ ]:
text_to_indices("Hello How are You")

In [ ]:
sequences = tokens.apply(text_to_indices)

In [ ]:
sequences.head()

In [ ]:
tensor_sequences = [torch.tensor(seq, dtype=torch.long) for seq in sequences]

In [ ]:
padded_tensor = pad_sequence(
    tensor_sequences,
    batch_first=True,
    padding_value=0
)

In [ ]:
padded_tensor.shape

In [ ]:
x = padded_tensor

In [ ]:
x[:1]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
      return self.X.shape[0]

    def __getitem__(self, idx):
       return self.X[idx], self.y[idx]

In [ ]:
dataset = CustomDataset(X_train, y_train)

In [ ]:
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        # Max pooling
        x = torch.max(output, dim=1).values

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model = LSTMModel(len(vocab), 128, 128, 2)
model.to(device)

In [ ]:
epochs = 50
learning_rate = 0.001

criterions = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

In [ ]:
for epoch in range(epochs):
  total_loss = 0
  for batch_x, batch_y in data_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    # Model predict
    output = model(batch_x)

    # Loss Calculate
    loss = criterions(output, batch_y)

    # backward propogation
    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss/len(data_loader)}")

In [ ]:
# ✅ Save best model
torch.save(model.state_dict(), "lstm_model.pth")

In [ ]:
model.eval()

In [ ]:
# test data evulation

test_dataset = CustomDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

correct = 0
total = 0

model.eval()

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        outputs = model(batch_x)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")